# Historical Data & Transaction Tape — Yagnum Research

**Purpose**: Learn how to pull historical price data and transaction lists for Solana assets.
**Cost**: $0 — using free tier APIs.

| Step | What You Learn | Data Source | Key Required? |
|------|---------------|-------------|---------------|
| 1 | Pull real-time aggregated pair data | DexScreener | No |
| 2 | Pull the transaction tape (list of trades) | Birdeye | Yes (Free) |

---
## The Separation of Concerns in DeFi
- **Jupiter**: Routes and executes the trade (The Broker/Exchange).
- **Solana**: Records the encrypted/raw byte data (The Ledger).
- **DexScreener / Birdeye**: Reads the ledger and organizes it into human-readable tables and charts (The Indexer).


## Step 0 — Dependencies


In [1]:
!pip install httpx python-dotenv --quiet

import httpx
import json
import os
from datetime import datetime
from dotenv import load_dotenv

print("All imports OK!")
NVDAX = 'Xsc9qvGR1efVDFGLrVsmkzv3qi45LTBjeUKSPmx9qEh'


All imports OK!



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1 — DexScreener (No API Key Required)

DexScreener provides a **completely free, public API** to get current pair statistics (price, liquidity, volume, FDV).
This is great for quick snapshots.

*Docs: https://docs.dexscreener.com/api/reference*


In [2]:
# 1. Fetch pair data from DexScreener
print("Fetching DexScreener data for NVDAx...")

with httpx.Client() as c:
    # DexScreener lets you search directly by the token's Mint Address!
    res = c.get(f"https://api.dexscreener.com/latest/dex/tokens/{NVDAX}")
    
    if res.status_code == 200:
        data = res.json()
        pairs = data.get("pairs", [])
        
        if pairs:
            # The API returns all liquidity pools containing this token. 
            # We usually just look at the most liquid one (the first one).
            main_pool = pairs[0]
            
            print(f"\n--- DEXSCREENER STATS ---")
            print(f"Token:       {main_pool['baseToken']['name']} ({main_pool['baseToken']['symbol']})")
            print(f"DEX:         {main_pool['dexId']}")
            print(f"Price:       ${float(main_pool['priceUsd']):.4f}")
            print(f"Liquidity:   ${float(main_pool['liquidity']['usd']):,.2f}")
            print(f"Volume(24h): ${float(main_pool['volume']['h24']):,.2f}")
            print(f"URL:         {main_pool['url']}")
        else:
            print("No pools found.")
    else:
        print("HTTP Error:", res.status_code)


Fetching DexScreener data for NVDAx...

--- DEXSCREENER STATS ---
Token:       NVIDIA xStock (NVDAx)
DEX:         orca
Price:       $224.9600
Liquidity:   $130,606.85
Volume(24h): $80,793.59
URL:         https://dexscreener.com/solana/6r4r93v5fcmzc13cl2eneepdsycr4qx3ptzbdwudtxco


## Step 2 — Birdeye: The Transaction Tape (Requires Free Key)

If you need the **actual list of historical transactions** (e.g., "Wallet X bought 5 NVDAx at 10:04 AM for $225.00"), you need an indexer like **Birdeye**.

1. Go to **[bds.birdeye.so](https://bds.birdeye.so/)**
2. Sign up and get a free Developer API key.
3. Paste it below.


In [3]:
BIRDEYE_API_KEY = ""

load_dotenv("../.env")
if not BIRDEYE_API_KEY:
    BIRDEYE_API_KEY = os.getenv("BIRDEYE_API_KEY", "")

if BIRDEYE_API_KEY:
    print("✅ Birdeye API Key loaded!")
else:
    print("⚠️ No Birdeye API Key found. The next cell will fail until you add one.")


✅ Birdeye API Key loaded!


### Fetching the Trade History

We will use the Birdeye `defi/txs/token` endpoint to get the last 50 trades for NVDAx.


In [4]:
# 2. Fetch Transaction Tape from Birdeye
if not BIRDEYE_API_KEY:
    print("Skipping: Please add your Birdeye API key above.")
else:
    print("Fetching last 10 trades from Birdeye...")
    
    headers = {
        "X-API-KEY": BIRDEYE_API_KEY,
        "x-chain": "solana"
    }
    
    with httpx.Client() as c:
        # offset=0 gets the most recent trades. limit=10 gets the last 10.
        res = c.get(
            f"https://public-api.birdeye.so/defi/txs/token?address={NVDAX}&offset=0&limit=10",
            headers=headers
        )
        
        if res.status_code == 200:
            data = res.json().get("data", {})
            items = data.get("items", [])
            
            print(f"\n{'Time (UTC)':<22} | {'Side':<5} | {'Size':<10} | {'Price (USD)':<12} | {'Wallet'}")
            print("-" * 80)
            
            for tx in items:
                # Convert unix timestamp to readable time
                dt = datetime.utcfromtimestamp(tx.get('blockUnixTime', 0)).strftime('%Y-%m-%d %H:%M:%S')
                
                side = tx.get('side', 'N/A').upper()
                price = tx.get('tokenPrice', 0)
                
                # The 'base' object represents the token we queried (NVDAx)
                base = tx.get('base', {})
                size = base.get('uiAmount', 0)
                
                wallet = tx.get('owner', 'Unknown')[:8] + "..."
                
                print(f"{dt:<22} | {side:<5} | {size:<10.4f} | ${price:<11.4f} | {wallet}")
                
        else:
            print(f"HTTP Error {res.status_code}: {res.text}")


Fetching last 10 trades from Birdeye...

Time (UTC)             | Side  | Size       | Price (USD)  | Wallet
--------------------------------------------------------------------------------
2026-08-16 01:31:04    | BUY   | 0.0000     | $224.6011    | 5KXDF6Qn...
2026-08-16 01:30:44    | BUY   | 0.0000     | $224.6011    | 5KXDF6Qn...
2026-08-16 01:28:57    | BUY   | 0.0012     | $224.6011    | 8bbUvfEx...
2026-08-16 01:28:53    | BUY   | 0.0011     | $224.6011    | 5KXDF6Qn...
2026-08-16 01:28:47    | BUY   | 0.0012     | $224.6011    | 8bbUvfEx...
2026-08-16 01:28:47    | BUY   | 0.0365     | $224.6011    | 5KXDF6Qn...
2026-08-16 01:28:45    | BUY   | 0.0010     | $224.6011    | FUEL3Te8...
2026-08-16 01:28:45    | SELL  | 0.0003     | $224.6011    | FUEL3Te8...
2026-08-16 01:28:45    | BUY   | 0.0001     | $224.6011    | 5KXDF6Qn...
2026-08-16 01:28:37    | BUY   | 0.0029     | $224.6011    | 5KXDF6Qn...


C:\Users\docker\AppData\Local\Temp\ipykernel_20344\949439649.py:28: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  dt = datetime.utcfromtimestamp(tx.get('blockUnixTime', 0)).strftime('%Y-%m-%d %H:%M:%S')


## Conclusion for Your Paper

For your empirical dataset:
1. **Jupiter** handles the actual bridging (ERR mechanics) via the Quote API.
2. **Birdeye** provides the historical dataset of the `P_JUP` equivalent prices and volumes over the weekends to prove the gap mechanics!

If you want to pull months of data, you simply write a loop that paginates the Birdeye API using the `offset` parameter.
